In [2]:
import pandas as pd
import json

In [69]:
def unify_dataset_schema(file_json, file_csv):
    # create df from json
    df1 = pd.read_json(file_json)
    df1 = df1.transpose()
    df2 = pd.read_csv(file_csv)
    # Initialize the new columns
    df1["Indication_approved_extracted"] = None
    df1["Indication_requested_extracted"] = None
    df1["Marketing_authorisation_holder_extracted"] = None
    
    for row in df1.iterrows():
        document_name = row[1].get("Document_name")
        if document_name in df2["Document_name"].values:
            matching_rows = df2[df2["Document_name"] == document_name]
            if not matching_rows.empty:
                matching_row = matching_rows.iloc[0]
                
                df1.loc[row[0], "Indication_approved_extracted"] = matching_row.get("Indication_approved_extracted", None)
                df1.loc[row[0], "Indication_requested_extracted"] = matching_row.get("Indication_requested_extracted", None)
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = matching_row.get("Marketing_authorisation_holder_extracted", None)
            else:
                # No matching rows found
                df1.loc[row[0], "Indication_approved_extracted"] = None
                df1.loc[row[0], "Indication_requested_extracted"] = None
                df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None
        else:
            # Document_name not in df2
            df1.loc[row[0], "Indication_approved_extracted"] = None
            df1.loc[row[0], "Indication_requested_extracted"] = None
            df1.loc[row[0], "Marketing_authorisation_holder_extracted"] = None

    df1 = df1.reindex(sorted(df1.columns), axis=1)

    return df1

# EMA

In [70]:
filepath_json = "./../inference/combined/EMA_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/EMA_manually_cleaned.csv"
merged_EMA = unify_dataset_schema(filepath_json, filepath_csv)
merged_EMA.head(1)

,Administration_route,Application_date,Application_year,Current_status,Decision,Decision_date,Decision_year,Disease_class(es),Document_name,Drug_class,...,Indication_requested_extracted,Marketing_authorisation_holder,Marketing_authorisation_holder_extracted,Marketing_authorisation_number,Non_proprietary_name,Nonclinical_abridged,Orphan_drug_status,Pharmaceutical_form,Procedure_number,Referral_body
febseltiq-withdrawal-assessment-report_en.pdf,oral,Not reported,Not reported,withdrawn,withdrawn,25.03.2022,2022,Neoplasms; Diseases of the digestive system,febseltiq-withdrawal-assessment-report_en.pdf,Small molecule,...,<Cholangiocarcinoma>,Not reported,Not Reported,EMA/939316/2022,infigratinib,no,yes,capsule,EMEA/H/C/005361/0000,NA


# Swissmedic

In [72]:
filepath_json = "./../inference/combined/SWISSMEDIC_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/SWISSMEDIC_manually_cleaned.csv"
merged_SWISSMEDIC = unify_dataset_schema(filepath_json, filepath_csv)
merged_SWISSMEDIC.head(1)

,Administration_route,Application_date,Application_year,Current_status,Decision,Decision_date,Decision_year,Disease_class(es),Document_name,Drug_class,...,Indication_requested_extracted,Marketing_authorisation_holder,Marketing_authorisation_holder_extracted,Marketing_authorisation_number,Non_proprietary_name,Nonclinical_abridged,Orphan_drug_status,Pharmaceutical_form,Procedure_number,Referral_body
swisspar_orladeyo.pdf,oral,22.07.2021,2021,authorised,approved,07.06.2022,2022,Diseases of the blood and blood-forming organs,swisspar_orladeyo.pdf,Small molecule,...,"<Angioedema, Hereditary>",BioCryst Schweiz GmbH,BioCryst,68464,berotralstat,yes,yes,capsule,EMEA/H/C/005138/0000,European Medicines Agency (EMA)


# Japan

In [73]:
filepath_json = "./../inference/combined/JAPAN_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/JAPAN_manually_cleaned.csv"
merged_JAPAN = unify_dataset_schema(filepath_json, filepath_csv)
merged_JAPAN.head(1)

,Administration_route,Application_date,Application_year,Current_status,Decision,Decision_date,Decision_year,Disease_class(es),Document_name,Drug_class,...,Indication_requested_extracted,Marketing_authorisation_holder,Marketing_authorisation_holder_extracted,Marketing_authorisation_number,Non_proprietary_name,Nonclinical_abridged,Orphan_drug_status,Pharmaceutical_form,Procedure_number,Referral_body
000245811.pdf,oral,28.02.2020,2020,authorised,approved,04.09.2020,2020,Neoplasms; Diseases of the genitourinary system,000245811.pdf,Small molecule,...,<Ovarian Neoplasms>,Takeda Pharmaceutical Company Limited,Takeda,Not reported,Niraparib,no,no,Capsule,Not reported,NA


# Australia

In [74]:
filepath_json = "./../inference/combined/AUSTRALIA_manually_cleaned.json"
filepath_csv = "./../inference/combined/with_extracted_data/Diseases_manually_cleaned/AUSTRALIA_manually_cleaned.csv"
merged_AUSTRALIA = unify_dataset_schema(filepath_json, filepath_csv)
merged_AUSTRALIA.head(1)

,Administration_route,Application_date,Application_year,Current_status,Decision,Decision_date,Decision_year,Disease_class(es),Document_name,Drug_class,...,Indication_requested_extracted,Marketing_authorisation_holder,Marketing_authorisation_holder_extracted,Marketing_authorisation_number,Non_proprietary_name,Nonclinical_abridged,Orphan_drug_status,Pharmaceutical_form,Procedure_number,Referral_body
auspar-eculizumab-201124.pdf,Intravenous,02.12.2019,2019,authorised (under additional monitoring),approved,26.06.2020,2020,Diseases of the nervous system,auspar-eculizumab-201124.pdf,Biologics,...,<Neuromyelitis Optica Spectrum Disorder>,Alexion Pharmaceuticals Australasia Pty Ltd,Alexion,138885,Eculizumab,yes,yes,solution,PM-2019-04825-1-1,Health Canada


# FDA

In [82]:
# not much to do, just maybe rename and merge with the rest
# apply the right json schema (ordering) and sort the csv columns
# take data\FDA\with_extracted_data_drug_class\FDA.json and extract "Non_proprietary_name_extracted": "Small molecule" to add as drug class
filepath1 = "./../inference/combined/FDA_manually_cleaned.json"
filepath2 = "./../data/FDA/with_extracted_data_drug_class/FDA.json"
filepath3 = "./data/datasets/FDA.json"

with open(filepath2, "r", encoding="utf-8") as f:
    data = json.load(f)
    drug_class = data.get("Non_proprietary_name_extracted", "")
    # add to filepath1 as Drug_class
    with open(filepath1, "r", encoding="utf-8") as f:
        data = json.load(f)
        data["Drug_class"] = drug_class
    with open(filepath3, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)


FileNotFoundError: [Errno 2] No such file or directory: './data/datasets/FDA.json'

# HealthCanada

In [12]:
# waiting for data extraction with LLM